In [1]:
import sys
sys.path.append('..')

from src.data_pipeline import parse_transcript

In [2]:
import pandas as pd

df = pd.read_csv('../data/raw/gt1_8kElo_all.csv', nrows=10)
df[['WhiteElo', 'BlackElo', 'Result', 'transcript']]

,WhiteElo,BlackElo,Result,transcript
0,1829,1857,0-1,1.d4 c5 2.c3 d5 3.e3 Nc6 4.Nf3 e6 5.Be2 Nf6 6....
1,1959,1970,0-1,1.e4 c5 2.Nf3 Nc6 3.Bc4 h6 4.Nc3 e5 5.d3 Nf6 6...
2,2036,2073,1-0,1.c4 Nf6 2.d4 d5 3.cxd5 Nxd5 4.Nf3 g6 5.e4 Nb6...
3,1890,1889,1/2-1/2,1.e4 c5 2.Nf3 b6 3.Bb5 Bb7 4.Nc3 a6 5.Be2 Nf6 ...
4,2070,1918,1-0,1.d4 d5 2.Nf3 Nc6 3.e3 Bg4 4.h3 Bh5 5.Be2 e6 6...
5,1873,1864,0-1,1.d4 c5 2.d5 e6 3.c4 exd5 4.cxd5 Nf6 5.Nc3 d6 ...
6,2025,1976,0-1,1.d4 d5 2.c4 c6 3.Nc3 Nf6 4.e3 e6 5.Nf3 Be7 6....
7,2110,2134,1-0,1.e4 c5 2.d4 cxd4 3.c3 d5 4.exd5 Nf6 5.cxd4 Nx...
8,2057,2073,1-0,1.d4 Nf6 2.c4 g6 3.Nc3 Bg7 4.e4 d6 5.f4 O-O 6....
9,1966,1998,0-1,1.e4 e6 2.f4 d5 3.e5 Qd7 4.Nf3 b6 5.Nc3 Ba6 6....


In [3]:
for transcript in df['transcript']:
    moves = parse_transcript(transcript)
    print(moves[:50], '...', len(moves), 'moves total')

['d4', 'c5', 'c3', 'd5', 'e3', 'Nc6', 'Nf3', 'e6', 'Be2', 'Nf6', 'O-O', 'Be7', 'Nbd2', 'a6', 'Qc2', 'b5', 'a3', 'c4', 'b3', 'Bb7', 'b4', 'Rc8', 'a4', 'Qc7', 'a5', 'e5', 'e4', 'exd4', 'cxd4', 'Nxb4', 'Qc3', 'dxe4', 'Ne5', 'Nbd5', 'Qg3', 'O-O', 'Ba3', 'Bxa3', 'Rxa3', 'b4', 'Raa1', 'c3', 'Nb3', 'c2', 'Nc5', 'Nc3', 'Bg4', 'Nxg4', 'Qxg4', 'f5'] ... 92 moves total
['e4', 'c5', 'Nf3', 'Nc6', 'Bc4', 'h6', 'Nc3', 'e5', 'd3', 'Nf6', 'O-O', 'Be7', 'h3', 'd6', 'Nd5', 'O-O', 'c3', 'Kh8', 'd4', 'cxd4', 'cxd4', 'exd4', 'Nxd4', 'Nxe4', 'Re1', 'f5', 'f3', 'Nxd4', 'Qxd4', 'Bf6', 'Nxf6', 'Qxf6', 'Qd5', 'Nc5', 'Be3', 'Nd7', 'Bd4', 'Qg6', 'Re7', 'Nf6', 'Qxd6', 'f4', 'Rae1', 'Bxh3', 'Bf1', 'Rad8', 'Qe5', 'Rxd4', 'Qxd4', 'Nh5'] ... 76 moves total
['c4', 'Nf6', 'd4', 'd5', 'cxd5', 'Nxd5', 'Nf3', 'g6', 'e4', 'Nb6', 'Be3', 'Bg7', 'Qd2', 'O-O', 'Bh6', 'Bg4', 'Bxg7', 'Kxg7', 'd5', 'Bxf3', 'gxf3', 'e5', 'Nc3', 'N8d7', 'O-O-O', 'Re8', 'h4', 'h5', 'f4', 'exf4', 'Be2', 'Qf6', 'Rdg1', 'Kh7', 'Rg5', 'Re5', 'Rhg1', 'Rxg

In [4]:
from src.config import get_config
from src.data_pipeline import build_splits, get_or_build_tokenizer, iter_move_sequences

In [6]:
config = get_config()
config

{'seed': 561,
 'raw_data_path': 'data/raw/gt1_8kElo_all.csv',
 'processed_data_dir': 'data/processed',
 'train_split': 0.9,
 'val_split': 0.05,
 'test_split': 0.05,
 'csv_chunksize': 100000,
 'tokenizer_file': 'data/tokenizer/tokenizer_chess.json',
 'min_frequency': 1,
 'vocab_size': 1000000}

In [7]:
import shutil
from pathlib import Path

Path('../data/sample').mkdir(exist_ok=True)
with open('../data/raw/gt1_8kElo_all.csv') as src, open('../data/sample/sample.csv', 'w') as dst:
    for i, line in enumerate(src):
        dst.write(line)
        if i >= 5000:  # header + 5000 rows
            break

sample_config = dict(config)
sample_config['raw_data_path'] = '../data/sample/sample.csv'
sample_config['processed_data_dir'] = '../data/sample/processed'
sample_config['tokenizer_file'] = '../data/sample/tokenizer_chess.json'
sample_config['csv_chunksize'] = 500

train_path, val_path, test_path = build_splits(sample_config)
tokenizer = get_or_build_tokenizer(sample_config)

Move vocabulary size: 3037.


In [8]:
import pandas as pd
print(len(pd.read_csv(train_path)), len(pd.read_csv(val_path)), len(pd.read_csv(test_path)))

sample_seq = next(iter_move_sequences(train_path, sample_config['csv_chunksize']))
print(sample_seq[:80])

enc = tokenizer.encode(sample_seq)
print(enc.tokens[:10], enc.ids[:10])
print(tokenizer.decode(enc.ids)[:80])  # should round-trip back to the same moves

4507 247 246
d4 c5 c3 d5 e3 Nc6 Nf3 e6 Be2 Nf6 O-O Be7 Nbd2 a6 Qc2 b5 a3 c4 b3 Bb7 b4 Rc8 a4 
['d4', 'c5', 'c3', 'd5', 'e3', 'Nc6', 'Nf3', 'e6', 'Be2', 'Nf6'] [6, 13, 30, 9, 46, 12, 5, 15, 48, 7]
d4 c5 c3 d5 e3 Nc6 Nf3 e6 Be2 Nf6 O-O Be7 Nbd2 a6 Qc2 b5 a3 c4 b3 Bb7 b4 Rc8 a4 


In [9]:
enc_unk = tokenizer.encode(sample_seq + " NotARealMove123")
print(enc_unk.tokens[-3:], enc_unk.ids[-3:])  # last token should be [UNK]
print([tokenizer.token_to_id(t) for t in ['[UNK]', '[PAD]', '[SOS]', '[EOS]']])

['Kf3', 'Rf2#', '[UNK]'] [89, 2104, 0]
[0, 1, 2, 3]
